# 07 — q99 pre-shock HMM taxonomy

## Research question

Conditional on a native five-minute q99 volatility shock occurring at (t_0), do the **24 candles strictly before it** contain several recurring precursor trajectories?

Every asset-level q99 onset is treated as its own instance. The HMM sees only:

\[
t_0-120m,\ t_0-115m,\ldots,\ t_0-5m.
\]

\[
\boxed{t_0\text{ is never an HMM emission.}}
\]

This is not another q99 forecasting experiment. Modules 06b/06c already found that their full-chronology HMM states did not improve q99 forecasting beyond a simple volatility benchmark. Module 07 asks a different conditional question: **given that the shock happened, are the pre-shock paths themselves heterogeneous?**


## Two latent objects

1. **HMM state** = a recurring market configuration inside a pre-shock window.
2. **Shock type** = a cluster of the complete 24-bar HMM posterior trajectory.

The HMM is therefore used as a temporal representation of each shock's precursor path. It is not assumed that one HMM state equals one shock type.

Shock sign, event magnitude, synchronization, prior-q99/aftershock status, asset and date are **not used in fitting**. They are examined only after the unsupervised taxonomy has been formed.


In [ ]:
from pathlib import Path
import json, os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 340)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_hmm_states_b as hb
from data_engineering.modeling import volatility_q99_preshock_hmm as v7

START = os.getenv('VD7_START', '').strip() or None
END = os.getenv('VD7_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD7_ASSETS', '').split(',') if x.strip()]
DISCOVERY_FRACTION = float(os.getenv('VD7_DISCOVERY_FRACTION', str(v7.DEFAULT_DISCOVERY_FRACTION)))
K_VALUES = tuple(int(x) for x in os.getenv('VD7_K', '2,3,4,5').split(',') if x.strip())
N_INIT = int(os.getenv('VD7_N_INIT', str(v7.DEFAULT_N_INIT)))
MAX_ITER = int(os.getenv('VD7_MAX_ITER', str(v7.DEFAULT_MAX_ITER)))
CLUSTER_K = tuple(int(x) for x in os.getenv('VD7_CLUSTER_K', '2,3,4').split(',') if x.strip())
CLUSTER_N_INIT = int(os.getenv('VD7_CLUSTER_N_INIT', str(v7.DEFAULT_CLUSTER_N_INIT)))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '07'
OUT.mkdir(parents=True, exist_ok=True)

print('Module 07 engine:', v7.VERSION)
print('selftest:', v7.selftest())
print('HMM K grid:', K_VALUES, '| shock-type grid:', CLUSTER_K)


## 1. Frozen −95m feature families

The source fields are the features carried by the Module 05 motif study. To avoid letting correlated representations manufacture HMM states, they are aggregated into four normalized family emissions:

- **activity:** intrabar range, USD volume, trade count;
- **IV level:** ATM forward IV, 25d put IV, 25d call IV;
- **liquidity/book activity:** spread, book update count;
- **L2 instability:** OBI zero-cross count/rate, transition count, path roughness.

The L2 family may be missing before L2 collection begins; the HMM marginalizes missing emission dimensions rather than discarding the entire sequence.


In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
schema = con.execute('DESCRIBE time_series_feature_state').df()
available = set(schema.column_name.astype(str))
contract = v7.feature_contract(available)
display(contract)
missing = contract.loc[~contract.present, 'feature'].tolist()
if missing:
    raise RuntimeError(f'Missing Module 07 source features: {missing}')


## 2. Load the common five-minute panel


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

cols = ['state_time','underlying','return_5m'] + list(v7.SOURCE_FEATURES)
where = []
if START:
    ts = pd.Timestamp(START)
    ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
    where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
if END:
    ts = pd.Timestamp(END)
    ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
    end_ts = ts + pd.Timedelta(days=1)
    where.append(f"state_time < TIMESTAMPTZ '{end_ts.isoformat()}'")
if ASSETS:
    where.append('underlying IN (' + ','.join(repr(x) for x in ASSETS) + ')')
where_sql = '' if not where else ' WHERE ' + ' AND '.join(where)
sql = 'SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state' + where_sql + ' ORDER BY underlying,state_time'
df = con.execute(sql).df()
con.close()
df['state_time'] = pd.to_datetime(df.state_time, utc=True)
df = df.reset_index(drop=True)
if df.empty:
    raise RuntimeError('No rows found.')
if df.duplicated(['underlying','state_time']).any():
    raise RuntimeError('Duplicate asset/timestamp rows found.')
grid = vb.build_grid(df)
print(f'rows {len(df):,} | assets {grid.assets}')
print(df.state_time.min(), '->', df.state_time.max())


## 3. Recompute all q99 onsets and extract exact pre-shock windows

The event definition matches the original Modules 03–05: full-sample per-asset q99 of native (r_t^2), with consecutive q99 bars reduced to run onsets.

For every onset we extract exactly 24 rows at offsets (-24\ldots-1). The event row at offset 0 is retained only in the catalog for post-hoc shock description.


In [ ]:
events, q99_thresholds, tail, onset = v7.event_catalog(grid)
windows, window_audit = v7.exact_preshock_windows(grid, events)
events = events.merge(window_audit[['event_id','exact_24bar_window','n_rows_found']], on='event_id', how='left')
usable_mask = events.exact_24bar_window.to_numpy(bool)
events_u = events.loc[usable_mask].reset_index(drop=True)
windows_u = [w for w, ok in zip(windows, usable_mask) if ok]
if len(events_u) < 30:
    raise RuntimeError('Too few complete q99 pre-shock windows.')
print('all q99 asset-events:', len(events), '| usable exact 24-bar windows:', len(events_u))
print('unique usable q99 timestamps:', events_u.event_time.nunique())
display(events_u[['underlying','event_time','variance_threshold_multiple','event_direction','n_assets_q99_same_time','has_prior_q99_bar_120m']].head(30))


## 4. Chronological discovery/validation split

Shock timestamps, not asset-event rows, determine the split. All assets sharing a timestamp therefore remain on the same side.

The source-feature normalization is estimated only from panel rows before the first validation shock timestamp. It uses the hour-level asset×UTC-hour normalization supported by the 02b clock study rather than noisy exact five-minute slots.


In [ ]:
split_time, is_discovery, is_validation = v7.split_events_by_unique_time(events_u, DISCOVERY_FRACTION)
events_u['sample'] = np.where(is_discovery, 'discovery', 'validation')
print('validation starts:', split_time)
display(events_u.groupby('sample').agg(asset_events=('event_id','size'), unique_times=('event_time','nunique'), assets=('underlying','nunique')))

source_E = v7.source_emissions(grid)
normalizer_train_mask = grid.time_ns < pd.Timestamp(split_time).value
normalizer = hb.fit_hour_normalizer(grid, source_E, normalizer_train_mask)
source_Z = hb.apply_normalizer(grid, source_E, normalizer)
family_Z, family_names = v7.family_scores_from_source_z(source_Z, source_E.columns)
family_Z_spread_only, _ = v7.family_scores_from_source_z(source_Z, source_E.columns, spread_only_liquidity=True)
print('families:', family_names)


## 5. Window coverage audit

A pre-shock row can still have one missing family. Missing dimensions are marginalized by the HMM. This cell makes the available information explicit rather than silently imputing it.


In [ ]:
coverage_rows = []
for ev, seq in zip(events_u.itertuples(), windows_u):
    for j, fam in enumerate(family_names):
        finite = np.isfinite(family_Z[seq, j])
        coverage_rows.append({
            'event_id': int(ev.event_id), 'sample': ev.sample, 'underlying': ev.underlying,
            'event_time': ev.event_time, 'family': fam,
            'n_finite_bars': int(finite.sum()), 'coverage_24bars': float(finite.mean()),
        })
coverage = pd.DataFrame(coverage_rows)
display(coverage.groupby(['sample','family']).coverage_24bars.describe().round(3))


## 6. Fit the pre-shock HMM

Each 24-bar event window is one HMM sequence. Simultaneous asset-events receive weights (1/n\) so one broad crypto shock has total fitting weight one rather than up to seven.

The HMM state count is selected by discovery-sample BIC with a state-occupancy guard. Validation sequence likelihood is reported only as a stability diagnostic.


In [ ]:
disc_events = events_u.loc[is_discovery].reset_index(drop=True)
val_events = events_u.loc[is_validation].reset_index(drop=True)
disc_windows = [w for w, m in zip(windows_u, is_discovery) if m]
val_windows = [w for w, m in zip(windows_u, is_validation) if m]
disc_seq = [family_Z[w] for w in disc_windows]
val_seq = [family_Z[w] for w in val_windows]
disc_w = v7.event_weights(disc_events)
val_w = v7.event_weights(val_events)

hmm_compare, hmm_models = v7.hmm_model_grid(
    disc_seq, disc_w, val_seq, val_w,
    k_values=K_VALUES, n_init=N_INIT, max_iter=MAX_ITER,
)
selected_k = int(hmm_compare.loc[hmm_compare.selected_by_bic, 'K'].iloc[0])
model = hmm_models[selected_k]
print('selected pre-shock HMM K:', selected_k)
display(hmm_compare.round(5))


## 7. What do the pre-shock HMM states represent?

These are unlabeled phases. Their emission means are robust-z family levels relative to the asset/hour training baseline.


In [ ]:
hmm_emissions = v7.hmm_emission_table(model, family_names)
hmm_transitions = v7.hmm_transition_table(model)
display(hmm_emissions.pivot(index='state', columns='family', values='mean_robust_z').round(3))
display(hmm_transitions.pivot(index='from_state', columns='to_state', values='probability').round(3))


## 8. Encode every shock by its entire posterior state trajectory

The HMM is smoothed **within the pre-shock window only**. This is acceptable for retrospective taxonomy because all 24 observations are known by (t_0-5m); no (t_0) or post-shock observation is used.

Each shock embedding contains six consecutive 20-minute state-occupancy blocks, plus the exact −95m posterior and the terminal −5m posterior.


In [ ]:
disc_paths, disc_embed_raw = v7.posterior_paths(model, disc_seq, disc_events, sample='discovery')
val_paths, val_embed_raw = v7.posterior_paths(model, val_seq, val_events, sample='validation')
disc_embed, val_embed, embed_mu, embed_sd = v7.standardize_embeddings(disc_embed_raw, val_embed_raw)
print('embedding dimension:', disc_embed.shape[1])


## 9. Discover candidate shock types

Shock types are weighted k-means clusters of the discovery posterior-trajectory embeddings. Candidate counts are 2–4. Selection uses weighted silhouette with a minimum effective cluster size of five unique-timestamp equivalents.

Validation shocks are assigned to the frozen discovery centroids; they do not refit the taxonomy.


In [ ]:
cluster_compare, n_types, disc_labels, val_labels, centers = v7.fit_shock_clusters(
    disc_embed, disc_w, val_embed,
    cluster_values=CLUSTER_K, n_init=CLUSTER_N_INIT,
)
print('selected number of shock types:', n_types)
display(cluster_compare.round(4))

disc_assign = v7.assignment_table(disc_events, disc_labels, 'discovery')
val_assign = v7.assignment_table(val_events, val_labels, 'validation')
assignments = pd.concat([disc_assign, val_assign], ignore_index=True)
display(assignments.head(30))


## 10. Are the types different in their actual precursor feature paths?

The shock labels were formed from HMM posterior trajectories, not from shock outcomes. This cell maps each type back to the four observed family paths from −120m to −5m.


In [ ]:
family_profiles = pd.concat([
    v7.family_profile_table(family_Z, grid, disc_windows, disc_assign, family_names),
    v7.family_profile_table(family_Z, grid, val_windows, val_assign, family_names),
], ignore_index=True)
for sample in ['discovery','validation']:
    for fam in family_names:
        g = family_profiles[(family_profiles['sample']==sample) & (family_profiles.family==fam)]
        if g.empty:
            continue
        fig, ax = plt.subplots(figsize=(9,3))
        for typ, h in g.groupby('shock_type'):
            h = h.sort_values('rel_min')
            ax.plot(h.rel_min, h.mean_family_z, marker='o', markersize=2, label=f'type {typ}')
        ax.axvline(-95, linestyle='--', linewidth=1)
        ax.axhline(0, linewidth=1)
        ax.set_title(f'{sample}: {fam}')
        ax.set_xlabel('minutes before q99 shock')
        ax.set_ylabel('mean robust-z family score')
        ax.legend()
        plt.show()


## 11. HMM-state trajectories by shock type


In [ ]:
disc_type_map = dict(zip(disc_assign.event_id.astype(int), disc_assign.shock_type.astype(int)))
val_type_map = dict(zip(val_assign.event_id.astype(int), val_assign.shock_type.astype(int)))
disc_paths['shock_type'] = disc_paths.event_id.astype(int).map(disc_type_map)
val_paths['shock_type'] = val_paths.event_id.astype(int).map(val_type_map)
all_paths = pd.concat([disc_paths, val_paths], ignore_index=True)
state_profiles = v7.state_profile_table(all_paths)
display(state_profiles.head(40).round(4))


## 12. Only now inspect what each shock type corresponds to

These descriptors were not used in the HMM or clustering. They tell us whether an unsupervised precursor type later turns out to correspond to:

- larger/smaller q99 shocks;
- upward/downward shocks;
- synchronized market-wide shocks;
- recent-q99/aftershock sequences;
- a particular asset;
- a particular date or the post-L2 collector era.

A type dominated by one date/asset/collector era is a warning sign, not an economic discovery.


In [ ]:
descriptors = v7.shock_type_descriptors(assignments)
asset_composition, date_composition = v7.composition_tables(assignments)
display(descriptors.round(4))
display(asset_composition.sort_values(['sample','shock_type','share_within_type'], ascending=[True,True,False]).head(100).round(4))
display(date_composition.sort_values(['sample','shock_type','share_within_type'], ascending=[True,True,False]).head(100).round(4))


## 13. Does the taxonomy persist in later shocks?

Validation events are assigned to discovery centroids. A candidate type is more credible if it receives non-trivial validation weight and its validation trajectory remains close to the discovery centroid.


In [ ]:
stability = v7.validation_stability(
    disc_embed, val_embed, disc_labels, val_labels, centers, disc_w, val_w
)
display(stability.round(4))


## 14. Collector-shift sensitivity

Claude's 06b review flagged `book_update_count` as potentially affected by the 2026-10-04 collector change. We therefore rerun the entire HMM+taxonomy with the liquidity family defined from **spread only**, while keeping the other three families identical.

The Adjusted Rand Index (ARI) measures whether the event taxonomy survives this change without requiring cluster labels to line up numerically.


In [ ]:
disc_seq_s = [family_Z_spread_only[w] for w in disc_windows]
val_seq_s = [family_Z_spread_only[w] for w in val_windows]
sens_hmm_compare, sens_models = v7.hmm_model_grid(
    disc_seq_s, disc_w, val_seq_s, val_w,
    k_values=K_VALUES, n_init=N_INIT, max_iter=MAX_ITER, seed_base=7170,
)
sens_k = int(sens_hmm_compare.loc[sens_hmm_compare.selected_by_bic, 'K'].iloc[0])
sens_model = sens_models[sens_k]
sens_disc_paths, sens_disc_raw = v7.posterior_paths(sens_model, disc_seq_s, disc_events, sample='discovery')
sens_val_paths, sens_val_raw = v7.posterior_paths(sens_model, val_seq_s, val_events, sample='validation')
sens_disc, sens_val, _, _ = v7.standardize_embeddings(sens_disc_raw, sens_val_raw)
sens_cluster_compare, sens_n_types, sens_disc_labels, sens_val_labels, sens_centers = v7.fit_shock_clusters(
    sens_disc, disc_w, sens_val,
    cluster_values=CLUSTER_K, n_init=CLUSTER_N_INIT, seed_base=7870,
)
sensitivity = pd.DataFrame([{
    'primary_hmm_K': selected_k,
    'spread_only_hmm_K': sens_k,
    'primary_n_shock_types': n_types,
    'spread_only_n_shock_types': sens_n_types,
    'discovery_ARI': v7.adjusted_rand_index(disc_labels, sens_disc_labels),
    'validation_ARI': v7.adjusted_rand_index(val_labels, sens_val_labels),
}])
display(sensitivity.round(4))


## 15. Visualize individual shock paths

Each row below is one asset-level q99 event, sorted by assigned type and event time. Colors are HMM hard states inferred using only the 24 pre-shock bars. This is a diagnostic, not a claim that hard states are the optimal representation.


In [ ]:
for sample, paths, assn in [('discovery',disc_paths,disc_assign),('validation',val_paths,val_assign)]:
    order = assn.sort_values(['shock_type','event_time','underlying']).event_id.astype(int).tolist()
    mat = []
    for eid in order:
        h = paths[paths.event_id == eid].sort_values('rel_bar')
        if len(h) == 24:
            mat.append(h.hard_state.to_numpy(int))
    if mat:
        fig, ax = plt.subplots(figsize=(12, max(4, min(14, len(mat)*0.06))))
        ax.imshow(np.vstack(mat), aspect='auto', interpolation='nearest')
        ax.axvline(v7.MOTIF_INDEX, linewidth=1, linestyle='--')
        ax.set_xticks(range(0,24,3), [v7.PRE_BARS[i]*5 for i in range(0,24,3)])
        ax.set_xlabel('minutes before q99 shock')
        ax.set_ylabel('asset-level shock instances')
        ax.set_title(f'{sample}: pre-shock HMM hard-state paths, sorted by shock type')
        plt.show()


## 16. Persist outputs


In [ ]:
contract.to_csv(OUT / '07_feature_contract.csv', index=False)
events.to_csv(OUT / '07_q99_event_catalog_all.csv', index=False)
events_u.to_csv(OUT / '07_q99_event_catalog_usable.csv', index=False)
window_audit.to_csv(OUT / '07_window_audit.csv', index=False)
normalizer.to_csv(OUT / '07_source_hour_normalizer.csv', index=False)
coverage.to_csv(OUT / '07_family_window_coverage.csv', index=False)
hmm_compare.to_csv(OUT / '07_hmm_model_comparison.csv', index=False)
hmm_emissions.to_csv(OUT / '07_hmm_emissions.csv', index=False)
hmm_transitions.to_csv(OUT / '07_hmm_transitions.csv', index=False)
all_paths.to_csv(OUT / '07_event_state_paths.csv', index=False)
cluster_compare.to_csv(OUT / '07_shock_type_model_comparison.csv', index=False)
assignments.to_csv(OUT / '07_shock_assignments.csv', index=False)
family_profiles.to_csv(OUT / '07_shock_type_family_profiles.csv', index=False)
state_profiles.to_csv(OUT / '07_shock_type_state_profiles.csv', index=False)
descriptors.to_csv(OUT / '07_shock_type_descriptors.csv', index=False)
asset_composition.to_csv(OUT / '07_shock_type_asset_composition.csv', index=False)
date_composition.to_csv(OUT / '07_shock_type_date_composition.csv', index=False)
stability.to_csv(OUT / '07_validation_stability.csv', index=False)
sens_hmm_compare.to_csv(OUT / '07_sensitivity_hmm_comparison.csv', index=False)
sens_cluster_compare.to_csv(OUT / '07_sensitivity_shock_type_comparison.csv', index=False)
sensitivity.to_csv(OUT / '07_collector_sensitivity.csv', index=False)

embed_cols = [f'embedding_{i:02d}' for i in range(disc_embed_raw.shape[1])]
disc_emb_df = pd.DataFrame(disc_embed_raw, columns=embed_cols)
disc_emb_df.insert(0,'event_id',disc_events.event_id.to_numpy())
disc_emb_df['sample']='discovery'
val_emb_df = pd.DataFrame(val_embed_raw, columns=embed_cols)
val_emb_df.insert(0,'event_id',val_events.event_id.to_numpy())
val_emb_df['sample']='validation'
pd.concat([disc_emb_df,val_emb_df],ignore_index=True).to_csv(OUT / '07_event_embeddings.csv', index=False)

meta = {
    'version': v7.VERSION,
    'module': '07_q99_preshock_hmm_taxonomy',
    'question': 'Conditional on q99 occurring, are there distinct recurrent 120-minute pre-shock trajectories?',
    'event_definition': 'full-sample per-asset q99 native 5m variance onset, matching Modules 03-05',
    'hmm_window_minutes': [-120,-5],
    'hmm_window_bars': 24,
    'event_bar_t0_used_in_hmm': False,
    'post_event_data_used_in_hmm': False,
    'shock_outcomes_used_for_clustering': False,
    'family_emissions': family_names,
    'source_features': list(v7.SOURCE_FEATURES),
    'normalization': 'training/discovery-era asset x UTC hour robust z; source features transformed then averaged within Module05 families',
    'simultaneous_event_weighting': 'each unique q99 timestamp has total HMM/clustering weight 1 across assets',
    'hmm_selected_K': selected_k,
    'selected_n_shock_types': n_types,
    'discovery_validation_split_time': str(split_time),
    'taxonomy_validation': 'validation shocks assigned to frozen discovery centroids',
    'collector_sensitivity': 'full rerun with liquidity/book family reduced to spread only',
    'forecasting_claim': False,
    'regime_count_claim': False,
    'n_all_q99_asset_events': int(len(events)),
    'n_usable_q99_asset_events': int(len(events_u)),
    'n_usable_unique_event_times': int(events_u.event_time.nunique()),
}
(OUT / '07_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote Module 07 outputs to', OUT)


## Interpretation boundary

A useful result is **not** merely a positive silhouette. We want several things simultaneously:

- multiple shock types with non-trivial effective support;
- visibly different precursor-family and state trajectories;
- the same types receiving material support in the chronological validation shocks;
- no type dominated by one asset, one date, the L2-collector era, or prior-q99 aftershocks;
- reasonable agreement with the spread-only collector sensitivity.

If those conditions fail, the correct conclusion is that the apparent heterogeneity is not yet stable. If they hold, Module 07 will have identified a genuine candidate **taxonomy of q99 precursor paths**, which is a different result from forecasting q99 occurrence.
